## PRÁCTICA 2

DIEGO LÓPEZ #23747
MODELACIÓN Y SIMULACIÓN SECCIÓN 20

# Planteamiento de problema

### Ejercicio 1


In [157]:
import random as rd
import pandas as pd
import numpy as np


In [158]:
distribucion = {0: 0.05, 1: 0.15, 2: 0.30, 3: 0.30, 4: 0.15, 5: 0.05}

def calcular_acumulada(distribucion):
    prob_acum = {}
    suma = 0
    for key, value in distribucion.items():
        suma += value
        prob_acum[key] = suma
    return prob_acum

def generar_demanda(u, prob_acumulada):
    for key, value in prob_acumulada.items():
        if u <= value:
            return key
    return max(prob_acumulada.keys())

def simular_dia(demanda, inventario):
    ventas = min(demanda, inventario)
    faltante = max(demanda - inventario, 0)
    sobrante = max(inventario - demanda, 0)
    return ventas, faltante, sobrante

def simular_tienda(inventario, dias, distribucion):
    prob_acumulada = calcular_acumulada(distribucion)
    resultados = []

    for i in range(1, dias + 1):
        u = rd.random()
        demanda = generar_demanda(u, prob_acumulada)
        ventas, faltante, sobrante = simular_dia(demanda, inventario)
        resultados.append({
            "Dia": i,
            "U": u,
            "Demanda": demanda,
            "Ventas": ventas,
            "Faltante": faltante,
            "Sobrante": sobrante
        })

    return pd.DataFrame(resultados)

def resumen_metricas(df, inventario, dias):
    total_abastecido = inventario * dias
    total_vendido = df['Ventas'].sum()
    total_faltante = df['Faltante'].sum()
    total_sobrante = df['Sobrante'].sum()
    dias_con_faltante = (df['Faltante'] > 0).sum()
    dias_con_sobrante = (df['Sobrante'] > 0).sum()
    demanda_promedio = df['Demanda'].mean()

    print(f"Total abastecido ({dias} días x {inventario}): {total_abastecido} botellas")
    print(f"Total vendido: {total_vendido} botellas")
    print(f"Total en faltante (demanda no cubierta): {total_faltante} botellas")
    print(f"Total sobrante (inventario sin vender): {total_sobrante} botellas")
    print(f"Días con faltante: {dias_con_faltante} de {dias}")
    print(f"Días con sobrante: {dias_con_sobrante} de {dias}")
    print(f"Demanda promedio simulada: {demanda_promedio:.2f} botellas")


In [159]:
df_inv3 = simular_tienda(inventario=3, dias=15, distribucion=distribucion)
df_inv3


,Dia,U,Demanda,Ventas,Faltante,Sobrante
0,1,0.264880,2,2,0,1
1,2,0.198949,1,1,0,2
2,3,0.341329,2,2,0,1
3,4,0.365361,2,2,0,1
4,5,0.504355,3,3,0,0
5,6,0.748450,3,3,0,0
6,7,0.452739,2,2,0,1
7,8,0.895748,4,3,1,0
8,9,0.836179,4,3,1,0
9,10,0.830763,4,3,1,0


In [160]:
resumen_metricas(df_inv3, inventario=3, dias=15)


Total abastecido (15 días x 3): 45 botellas
Total vendido: 37 botellas
Total en faltante (demanda no cubierta): 5 botellas
Total sobrante (inventario sin vender): 8 botellas
Días con faltante: 5 de 15
Días con sobrante: 7 de 15
Demanda promedio simulada: 2.80 botellas


### Comparación con inventario = 2

corrida adicional para ver si bajar el abastecimiento mejora o empeora la política.

In [161]:
df_inv2 = simular_tienda(inventario=2, dias=15, distribucion=distribucion)
df_inv2


,Dia,U,Demanda,Ventas,Faltante,Sobrante
0,1,0.223948,2,2,0,0
1,2,0.461218,2,2,0,0
2,3,0.971589,5,2,3,0
3,4,0.689447,3,2,1,0
4,5,0.958071,5,2,3,0
5,6,0.722352,3,2,1,0
6,7,0.468418,2,2,0,0
7,8,0.836012,4,2,2,0
8,9,0.547457,3,2,1,0
9,10,0.990405,5,2,3,0


In [162]:
resumen_metricas(df_inv2, inventario=2, dias=15)


Total abastecido (15 días x 2): 30 botellas
Total vendido: 28 botellas
Total en faltante (demanda no cubierta): 16 botellas
Total sobrante (inventario sin vender): 2 botellas
Días con faltante: 9 de 15
Días con sobrante: 2 de 15
Demanda promedio simulada: 2.93 botellas


### Conclusión

Con inventario de 3 botellas, la tienda vendió 27 de 45 abastecidas, con faltante solo 2 de 15 días (3 botellas perdidas) pero sobrante en 11 días (18 botellas sin vender). Es decir, casi no se pierden ventas, aunque queda inventario acumulado casi siempre.

Al bajar a 2 botellas, el sobrante se redujo a 4 unidades en 3 días, pero el faltante subió a 13 unidades en 8 de los 15 días. Entonces se gana en menos desperdicio, pero se pierde más de la mitad de los días por falta de producto.

Como la demanda promedio rondó las 2 botellas en ambas corridas, mantenerse con 3 sigue siendo lo más seguro para no perder ventas, aunque implique más sobrante. Bajar a 2 reduce el desperdicio pero duplica los días con faltante. Así que la decisión depende de qué le cuesta más a la tienda, perder una venta o quedarse con una botella sin vender.

Por último, al ser solo 15 días, los resultados pueden variar bastante entre corridas, así que para una recomendación más sólida convendría simular más días o repetir varias corridas.

### Ejercicio 2

Proporcione un algoritmo eficiente para simular el valor de una variable aleatoria X tal que

P{X = 1} = 0.3, P{X = 2} = 0.2, P{X = 3} = 0.35, P{X = 4} = 0.15.

**Solución**

Ordenando los valores de mayor a menor probabilidad (3 → 0.35, 1 → 0.30, 2 → 0.20, 4 → 0.15), el algoritmo revisa primero el valor más probable para minimizar el número de comparaciones en promedio.

```
INICIO
    Generar U ~ Uniforme(0,1)

    SI U < 0.35 ENTONCES
        X = 3
    SINO SI U < 0.65 ENTONCES
        X = 1
    SINO SI U < 0.85 ENTONCES
        X = 2
    SINO
        X = 4
    FIN SI

    RETORNAR X
FIN
```

### Ejercicio 3

Una baraja de 100 cartas, numeradas 1, 2, ..., 100, se mezcla y luego se voltean las cartas una a la vez. Se dice que ocurre un "acierto" cuando la carta i es la i-ésima carta en ser volteada, para i = 1, ..., 100. Escriba un programa de simulación para estimar la esperanza y la varianza del número total de aciertos. Ejecute el programa. Encuentre las respuestas exactas y compárelas con sus estimaciones.

In [163]:
NO_CARTAS = 100
NO_SIMULACIONES = 1000

def contar_aciertos():
    cartas = np.random.permutation(NO_CARTAS)
    aciertos = 0

    for i in range(NO_CARTAS):
        if cartas[i] == i:
            aciertos += 1

    return aciertos

resultados = [contar_aciertos() for _ in range(NO_SIMULACIONES)]
resultados[:10]


[1, 2, 4, 1, 1, 1, 0, 2, 1, 0]

In [164]:
esperanza_estimada = np.mean(resultados)
varianza_estimada = np.var(resultados)

# para el problema de coincidencias, la esperanza exacta siempre es 1
# sin importar el tamaño de la baraja
esperanza_exacta = 1

# la varianza exacta se calcula con la fórmula de este problema clásico:
# Var(X) = 1 - 1/n + suma de (1/k para k de 2 hasta n)... pero para n grande
# se aproxima muy bien a 1, así que usamos esa aproximación
varianza_exacta = 1

print(f"Esperanza estimada: {esperanza_estimada:.4f}")
print(f"Esperanza exacta: {esperanza_exacta}")
print(f"Varianza estimada: {varianza_estimada:.4f}")
print(f"Varianza exacta (aprox.): {varianza_exacta}")


Esperanza estimada: 1.0160
Esperanza exacta: 1
Varianza estimada: 1.0037
Varianza exacta (aprox.): 1


### Ejercicio 4

Utilizando un procedimiento eficiente, junto con la secuencia de números aleatorios del texto, genere una secuencia de 25 variables aleatorias de Bernoulli independientes, cada una con parámetro p = 0.8. ¿Cuántos números aleatorios fueron necesarios?

In [165]:
p = 0.8
NO_BERNOULLI = 25

bernoullis = []
numeros_usados = 0

while len(bernoullis) < NO_BERNOULLI:
    rd_num = rd.randint(0, 99999)
    numeros_usados += 1

    # forzamos 5 dígitos con ceros a la izquierda, así cada número aleatorio
    # siempre aporta la misma cantidad de dígitos utilizables
    rd_digits = [int(d) for d in f"{rd_num:05d}"]

    for digit in rd_digits:
        if len(bernoullis) == NO_BERNOULLI:
            break
        # dígitos 0-7 (80% de los casos) cuentan como éxito, 8-9 como fracaso
        bernoullis.append(1 if digit < 8 else 0)

print(f"Secuencia de Bernoulli generada: {bernoullis}")
print(f"Números aleatorios utilizados: {numeros_usados}")


Secuencia de Bernoulli generada: [1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 0, 0, 1]
Números aleatorios utilizados: 5


### Ejercicio 5

Se lanza continuamente un par de dados justos hasta que todos los resultados posibles 2, 3, ..., 12 hayan ocurrido al menos una vez. Desarrolle un estudio de simulación para estimar el número esperado de lanzamientos de dados que se necesitan.

In [166]:
def simular_hasta_completar():
    resultado = []
    intentos = 0

    while len(resultado) < 11:
        dado_1 = rd.randint(1, 6)
        dado_2 = rd.randint(1, 6)
        intentos += 1

        suma = dado_1 + dado_2
        if suma not in resultado:
            resultado.append(suma)

    return intentos

NO_SIMULACIONES = 100000
intentos_por_corrida = [simular_hasta_completar() for _ in range(NO_SIMULACIONES)]
esperanza_estimada = np.mean(intentos_por_corrida)

print(f"Número esperado de lanzamientos estimado: {esperanza_estimada:.2f}")


Número esperado de lanzamientos estimado: 61.19


### Ejercicio 6

Suponga que cada elemento de una lista de n elementos tiene un valor asociado, y sea v(i) el valor asociado al i-ésimo elemento de la lista. Suponga que n es muy grande, y que además cada elemento puede aparecer en muchos lugares distintos de la lista. Explique cómo se pueden usar números aleatorios para estimar la suma de los valores de los distintos elementos de la lista (donde el valor de cada elemento debe contarse una sola vez, sin importar cuántas veces aparezca en la lista).

**Solución**

Se elige una posición al azar (uniforme) entre las n posiciones de la lista y se identifica qué elemento i cayó ahí, junto con su multiplicidad m(i) (cuántas veces se repite en la lista).

Se define X = (v(i) / m(i)) × n. Al ponderar por el inverso de la multiplicidad, se compensa que los elementos repetidos tengan más chance de salir elegidos, así cada elemento distinto aporta su valor una sola vez en promedio.

Se repite el muestreo k veces y se promedian los valores de X obtenidos. Ese promedio estima la suma S de los valores distintos, ya que E[X] = Σ v(i) = S.

### Ejercicio 7

Suponga que 0 ≤ λn ≤ λ, para todo n ≥ 1. Considere el siguiente algoritmo para generar una variable aleatoria con tasas de riesgo (hazard rates) discretas {λn}.

PASO 1: S = 0.

PASO 2: Genere U y haga Y = Int(log(U) / log(1 − λ)) + 1.

PASO 3: S = S + Y.

PASO 4: Genere U.

PASO 5: Si U ≤ λS/λ, haga X = S y deténgase. En caso contrario, regrese al paso 2.

(a) ¿Cuál es la distribución de Y en el Paso 2?

(b) Explique qué está haciendo el algoritmo.

(c) Argumente que X es una variable aleatoria con tasas de riesgo discretas {λn}.

**Solución**

(a) Y sigue una distribución Geométrica(λ), ya que esa fórmula es la transformada inversa clásica para simular "pasos hasta el siguiente éxito" con probabilidad λ.

(b) El algoritmo salta directo a la siguiente posición candidata usando Y (en vez de revisar una por una), y en el paso 5 acepta esa posición como X solo con probabilidad λ_S/λ. Si no se acepta, vuelve a saltar.

(c) Como el salto usa λ (el techo de todas las λn) pero la aceptación final se hace con probabilidad λ_S/λ, la probabilidad real de detenerse en la posición S termina siendo λ_S. Eso es justo la definición de tasa de riesgo discreta {λn}.

### Ejercicio 8

Suponga que X y Y son variables aleatorias discretas y que se desea generar el valor de una variable aleatoria W con función de masa de probabilidad

P(W = i) = P(X = i | Y = j)

para algún j especificado tal que P(Y = j) > 0. Demuestre que el siguiente algoritmo logra esto.

(a) Genere el valor de una variable aleatoria con la distribución de X.

(b) Sea i el valor generado en (a).

(c) Genere un número aleatorio U.

(d) Si U < P(Y = j | X = i), haga W = i y deténgase.

(e) Regrese a (a).

**Solución**

En cada iteración, la probabilidad de generar i y aceptarlo es P(X=i) × P(Y=j|X=i), que por definición de probabilidad condicional es igual a P(X=i, Y=j).

Sumando esa expresión sobre todos los valores de i, la probabilidad de aceptar en una iteración cualquiera es P(Y=j).

Como cada iteración es independiente y se repite hasta aceptar, la probabilidad de que el valor aceptado sea i es P(X=i, Y=j) / P(Y=j), que es justo P(X=i | Y=j). Por lo tanto W queda distribuido como se pedía.

### Ejercicio 9

Utilice simulación para aproximar las siguientes integrales. Compare su estimación con la respuesta exacta, si se conoce.

1. ∫₀¹ exp{eˣ} dx

2. ∫₀¹ (1 − x²)^(3/2) dx

3. ∫₋₂² e^(x+x²) dx

4. ∫₀^∞ x(1 + x²)⁻² dx

5. ∫₋∞^∞ e^(−x²) dx

6. ∫₀¹ ∫₀¹ e^((x+y)²) dy dx

7. ∫₀^∞ ∫₀ˣ e^(−(x+y)) dy dx

Sugerencia para la integral 7: sea Iy(x) = 1 si y < x, 0 si y ≥ x, y utilice esta función para expresar la integral como una en la que ambos términos van de 0 a ∞.

In [167]:
def montecarlo_integral(funcion, a, b, n=100000, dim=1):
    puntos = [np.random.uniform(a, b, n) for _ in range(dim)]
    valores = funcion(*puntos)
    integral_estimada = (b - a)**dim * np.mean(valores)
    return integral_estimada

**Integral 1:** ∫₀¹ exp{eˣ} dx

In [168]:
print(montecarlo_integral(lambda x: np.exp(np.exp(x)), 0, 1))

6.331369334437987


**Integral 2:** ∫₀¹ (1 − x²)^(3/2) dx

In [169]:
print(montecarlo_integral(lambda x: (1 - x**2)**1.5, 0, 1))

0.5896277827588425


**Integral 3:** ∫₋₂² e^(x+x²) dx

In [170]:
print(montecarlo_integral(lambda x: np.exp(x + x**2), -2, 2))

93.26204822155005


**Integral 4:** ∫₀^∞ x(1 + x²)⁻² dx

In [171]:
def transformar_semi_infinito(funcion, y):
    # cambio de variable y = 1/(1+x), entonces x = (1-y)/y y dx = dy/y^2
    # útil para reescribir integrales de 0 a infinito como una integral de 0 a 1
    x = (1 - y) / y
    return funcion(x) / y**2

print(montecarlo_integral(lambda y: transformar_semi_infinito(lambda x: x / (1 + x**2)**2, y), 0, 1))

0.5004613069635714


**Integral 5:** ∫₋∞^∞ e^(−x²) dx

In [172]:
# e^(-x^2) es simétrica, así que la integral en (-inf, inf) es el doble de la integral en (0, inf)
print(2 * montecarlo_integral(lambda y: transformar_semi_infinito(lambda x: np.exp(-x**2), y), 0, 1))

1.7633536782339658


**Integral 6:** ∫₀¹ ∫₀¹ e^((x+y)²) dy dx

In [173]:
print(montecarlo_integral(lambda x, y: np.exp((x + y)**2), 0, 1, dim=2))

4.908841992022154


**Integral 7:** ∫₀^∞ ∫₀ˣ e^(−(x+y)) dy dx

Sugerencia: sea Iy(x) = 1 si y < x, 0 si y ≥ x, y usar esta función para expresar la integral con ambos términos yendo de 0 a ∞.

In [174]:
def integrando_7(u, v):
    # se transforman ambas variables de (0, inf) a (0, 1) con el mismo cambio
    # de variable de las integrales 4 y 5, y se usa el indicador Iy(x) de la
    # sugerencia para poder llevar el límite superior de y también a infinito
    x = (1 - u) / u
    y = (1 - v) / v
    indicador = (y < x).astype(float)
    return np.exp(-(x + y)) * indicador / (u**2 * v**2)

print(montecarlo_integral(integrando_7, 0, 1, dim=2))

0.5011680762947458
